# Team 2 — CNN NILM (Kettle ON) — Clean Final Notebook (Upload-to-Colab)

This notebook is the cleaned final version. It:

- Uploads House 1 + House 2 kettle analysis CSVs into Colab
- Creates `kettle_on` from `kettle_watts` using a threshold **chosen from the data** (so it won't be impossible like thr=500 when max is 311)
- Builds windows from mains `power_watts`
- Trains CNN on **House 2** and evaluates on **House 1**
- Prints **classification report + confusion matrix** and saves artifacts

Data files you exported/uploaded had names like: `House_2_kettle_analysis - House_2_kettle_analysis.csv` and `House_1_kettle_analysis - House_1_kettle_analysis.csv`. [Source](https://www.genspark.ai/api/files/s/4ll0gfS1)

## 0) Install + imports

In [1]:
!pip -q install numpy pandas scikit-learn tensorflow

import os, json
from pathlib import Path
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix


## 1) Upload the two CSVs into Colab (select both files in the picker)

In [2]:
from google.colab import files

uploaded = files.upload()  # select BOTH house CSVs
print('Uploaded:', list(uploaded.keys()))

!ls -lah /content | grep -i -E 'house_1|house_2|kettle|analysis|csv'


Saving House_1_kettle_analysis - House_1_kettle_analysis.csv to House_1_kettle_analysis - House_1_kettle_analysis.csv
Saving House_2_kettle_analysis - House_2_kettle_analysis.csv to House_2_kettle_analysis - House_2_kettle_analysis.csv
Uploaded: ['House_1_kettle_analysis - House_1_kettle_analysis.csv', 'House_2_kettle_analysis - House_2_kettle_analysis.csv']
-rw-r--r-- 1 root root 239K Feb 12 17:41 House_1_kettle_analysis - House_1_kettle_analysis.csv
-rw-r--r-- 1 root root 246K Feb 12 17:41 House_2_kettle_analysis - House_2_kettle_analysis.csv


## 2) Auto-detect file paths (works even if Colab adds (1) to filenames)

In [3]:
import glob

h2 = sorted(glob.glob('/content/*House_2*kettle*analysis*.csv'))
h1 = sorted(glob.glob('/content/*House_1*kettle*analysis*.csv'))

print('House 2 candidates:', h2)
print('House 1 candidates:', h1)

if not h2 or not h1:
    raise FileNotFoundError('Could not find both House_1 and House_2 kettle analysis CSVs under /content/')

HOUSE2_CSV = h2[0]
HOUSE1_CSV = h1[0]

print('Using HOUSE2_CSV:', HOUSE2_CSV)
print('Using HOUSE1_CSV:', HOUSE1_CSV)


House 2 candidates: ['/content/House_2_kettle_analysis - House_2_kettle_analysis.csv']
House 1 candidates: ['/content/House_1_kettle_analysis - House_1_kettle_analysis.csv']
Using HOUSE2_CSV: /content/House_2_kettle_analysis - House_2_kettle_analysis.csv
Using HOUSE1_CSV: /content/House_1_kettle_analysis - House_1_kettle_analysis.csv


## 3) Load + basic cleaning

In [6]:
train_df = pd.read_csv(HOUSE2_CSV)
test_df  = pd.read_csv(HOUSE1_CSV)

for df in (train_df, test_df):
    df['index'] = pd.to_datetime(df['index'], utc=True, errors='coerce')
    df.dropna(subset=['index','power_watts','kettle_watts'], inplace=True)
    df.sort_values('index', inplace=True)
    df.reset_index(drop=True, inplace=True)

print('Train cols:', train_df.columns.tolist(), 'rows:', len(train_df))
print('Test  cols:', test_df.columns.tolist(), 'rows:', len(test_df))
print('House1 kettle_watts max:', float(test_df['kettle_watts'].max()))
print('House2 kettle_watts max:', float(train_df['kettle_watts'].max()))


Train cols: ['index', 'power_watts', 'kettle_watts'] rows: 5630
Test  cols: ['index', 'power_watts', 'kettle_watts'] rows: 5630
House1 kettle_watts max: 311.5633
House2 kettle_watts max: 1179.5862


## 4) Choose a reasonable threshold + create Boolean `kettle_on`

We pick a threshold that's **below max** but above typical noise.

Rule used:
- Compute 99th percentile of kettle_watts in TRAIN (House 2)
- Threshold = 0.6 × p99 (clipped to [50, 1000])

Then print class balance so you can adjust if needed.

In [8]:
p99 = float(np.percentile(train_df['kettle_watts'].to_numpy(), 99))
THRESHOLD_W = int(np.clip(0.6 * p99, 50, 1000))

train_df['kettle_on'] = (train_df['kettle_watts'] >= THRESHOLD_W).astype(int)
test_df['kettle_on']  = (test_df['kettle_watts']  >= THRESHOLD_W).astype(int)

print('p99(train kettle_watts)=', p99)
print('THRESHOLD_W=', THRESHOLD_W)
print('Train kettle_on counts:')
print(train_df['kettle_on'].value_counts(dropna=False))
print('Test kettle_on counts:')
print(test_df['kettle_on'].value_counts(dropna=False))

print('Top kettle events (House1):')
print(test_df.sort_values('kettle_watts', ascending=False).head(10)[['index','kettle_watts','power_watts','kettle_on']])


p99(train kettle_watts)= 315.1505182
THRESHOLD_W= 189
Train kettle_on counts:
kettle_on
0    5495
1     135
Name: count, dtype: int64
Test kettle_on counts:
kettle_on
0    5614
1      16
Name: count, dtype: int64
Top kettle events (House1):
                         index  kettle_watts  power_watts  kettle_on
525  2013-03-11 13:00:00+00:00     311.56330   462.202881          1
5089 2013-09-17 17:00:00+00:00     266.30328   457.008911          1
5011 2013-09-14 11:00:00+00:00     262.14800   511.139740          1
522  2013-03-11 10:00:00+00:00     256.19952   572.064270          1
2078 2013-05-15 06:00:00+00:00     238.50854   445.844788          1
2058 2013-05-14 10:00:00+00:00     227.76114   501.197144          1
4418 2013-08-20 18:00:00+00:00     227.73022   460.145111          1
3888 2013-07-29 16:00:00+00:00     216.30756   642.424927          1
1123 2013-04-05 11:00:00+00:00     205.68115   576.018005          1
5616 2013-10-09 16:00:00+00:00     203.53188   531.721435          1


## 5) Sampling interval → choose WINDOW/STRIDE

In [10]:
train_delta = train_df['index'].diff().value_counts().head(5)
print('Most common train deltas:', train_delta)

most_common = train_df['index'].diff().mode().iloc[0] if len(train_df) > 2 else pd.Timedelta(hours=1)

if most_common <= pd.Timedelta(minutes=2):
    WINDOW = 256
    STRIDE = 16
else:
    WINDOW = 24
    STRIDE = 1

print('Using WINDOW=', WINDOW, 'STRIDE=', STRIDE)


Most common train deltas: index
0 days 01:00:00    5629
Name: count, dtype: int64
Using WINDOW= 24 STRIDE= 1


## 6) Build windows (label rule: ANY-ON-IN-WINDOW)

This is better than end-of-window for short kettle events.

In [11]:
def make_windows_any_on(df, window, stride):
    x = df['power_watts'].astype('float32').to_numpy()
    y = df['kettle_on'].astype('int32').to_numpy()

    X, Y = [], []
    for end in range(window, len(df), stride):
        start = end - window
        X.append(x[start:end])
        Y.append(int(y[start:end].max()))

    X = np.array(X, dtype='float32')[..., None]
    Y = np.array(Y, dtype='int32')
    return X, Y

X_all, y_all = make_windows_any_on(train_df, WINDOW, STRIDE)
X_test, y_test = make_windows_any_on(test_df, WINDOW, STRIDE)

print('House2 windows:', X_all.shape, y_all.shape)
print('House1 windows:', X_test.shape, y_test.shape)
print('Train positive rate:', float(y_all.mean()))
print('Test positive rate:', float(y_test.mean()))


House2 windows: (5606, 24, 1) (5606,)
House1 windows: (5606, 24, 1) (5606,)
Train positive rate: 0.30806278986799857
Test positive rate: 0.06136282554405994


## 7) Train/Val split on House 2 + normalization (fit on House 2 train only)

In [12]:
VAL_FRAC = 0.15
N = len(X_all)
split = int(N * (1 - VAL_FRAC))

X_train, y_train = X_all[:split], y_all[:split]
X_val, y_val     = X_all[split:], y_all[split:]

mu = X_train.mean()
sd = X_train.std() + 1e-6

X_train_n = (X_train - mu) / sd
X_val_n   = (X_val   - mu) / sd
X_test_n  = (X_test  - mu) / sd

print('Train/Val/Test shapes:', X_train_n.shape, X_val_n.shape, X_test_n.shape)
print('Norm mu/sd:', float(mu), float(sd))


Train/Val/Test shapes: (4765, 24, 1) (841, 24, 1) (5606, 24, 1)
Norm mu/sd: 291.8094177246094 288.5320739746094


## 8) Class weights

In [13]:
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())

if pos == 0:
    raise ValueError('No positive examples after threshold/windowing. Lower THRESHOLD_W or inspect kettle_watts.')

w0 = 0.5 * (len(y_train) / neg)
w1 = 0.5 * (len(y_train) / pos)
class_weight = {0: w0, 1: w1}

print('neg:', neg, 'pos:', pos)
print('class_weight:', class_weight)


neg: 3242 pos: 1523
class_weight: {0: 0.7348858729179519, 1: 1.5643466841759686}


## 9) CNN model + train (logging/checkpoints)

In [14]:
def make_cnn(window):
    inp = keras.Input(shape=(window, 1))
    x = layers.Conv1D(32, 7, padding='same', activation='relu')(inp)
    x = layers.MaxPool1D(2)(x)
    x = layers.Conv1D(64, 5, padding='same', activation='relu')(x)
    x = layers.MaxPool1D(2)(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.BinaryAccuracy(name='acc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc'),
        ],
    )
    return model

model = make_cnn(WINDOW)
model.summary()

Path('reports').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)

run_name = f'cnn_house2_to_house1_kettle_on_thr{THRESHOLD_W}_anyOn'

callbacks = [
    keras.callbacks.CSVLogger(f'reports/{run_name}_history.csv', append=False),
    keras.callbacks.ModelCheckpoint(
        filepath=f'models/{run_name}.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    X_train_n, y_train,
    validation_data=(X_val_n, y_val),
    epochs=30,
    batch_size=128,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 24, 32)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 12, 64)         │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 6, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,585 (170.25 KB)

 Trainable params: 43,585 (170.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
35/38 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - acc: 0.4907 - auc: 0.6506 - loss: 0.6568 - precision: 0.3778 - recall: 0.9130
Epoch 1: val_auc improved from -inf to 0.51576, saving model to models/cnn_house2_to_house1_kettle_on_thr189_anyOn.keras
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 44ms/step - acc: 0.4956 - auc: 0.6531 - loss: 0.6539 - precision: 0.3800 - recall: 0.9090 - val_acc: 0.5077 - val_auc: 0.5158 - val_loss: 0.8047 - val_precision: 0.2775 - val_recall: 0.6422
Epoch 2/30
36/38 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - acc: 0.6211 - auc: 0.7581 - loss: 0.5703 - precision: 0.4550 - recall: 0.8492
Epoch 2: val_auc did not improve from 0.51576
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - acc: 0.6235 - auc: 0.7594 - loss: 0.5691 - precision: 0.4566 - recall: 0.8475 - val_acc: 0.5446 - val_auc: 0.4927 - val_loss: 0.9046 - val_precision: 0.2812 - val_recall: 0.5637
Epoch 3/30
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - acc: 0.7122 - auc: 0.8161 - loss: 0.5097 - precision: 0.5258 - recall: 0.8194


In [16]:
from sklearn.metrics import precision_recall_fscore_support

for thr in [0.5, 0.6, 0.7, 0.8, 0.9]:
    y_pred_thr = (y_prob >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred_thr, average="binary", zero_division=0)
    print(f"thr={thr:.1f}  precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}")


thr=0.5  precision=0.081  recall=0.485  f1=0.139
thr=0.6  precision=0.090  recall=0.355  f1=0.143
thr=0.7  precision=0.113  recall=0.265  f1=0.159
thr=0.8  precision=0.140  recall=0.169  f1=0.153
thr=0.9  precision=0.132  recall=0.041  f1=0.062


## 10) Evaluate on House 1 + save config

In [18]:
DECISION_THRESHOLD = 0.7
y_pred = (y_prob >= DECISION_THRESHOLD).astype(int)
print("Decision threshold:", DECISION_THRESHOLD)


print('Classification report:')
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

config = {
    'threshold_w': int(THRESHOLD_W),
    'p99_train_kettle_watts': float(p99),
    'window': int(WINDOW),
    'stride': int(STRIDE),
    'val_frac': float(VAL_FRAC),
    'label_rule': 'any_on_in_window',
    'norm_mu': float(mu),
    'norm_sd': float(sd),
    'house_train': 'House 2',
    'house_test': 'House 1',
}
with open(f'reports/{run_name}_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved:', f'reports/{run_name}_history.csv', f'models/{run_name}.keras', f'reports/{run_name}_config.json')


Decision threshold: 0.7
Classification report:
              precision    recall  f1-score   support

           0     0.9473    0.8647    0.9041      5262
           1     0.1133    0.2645    0.1587       344

    accuracy                         0.8279      5606
   macro avg     0.5303    0.5646    0.5314      5606
weighted avg     0.8961    0.8279    0.8584      5606

Confusion matrix:
[[4550  712]
 [ 253   91]]
Saved: reports/cnn_house2_to_house1_kettle_on_thr189_anyOn_history.csv models/cnn_house2_to_house1_kettle_on_thr189_anyOn.keras reports/cnn_house2_to_house1_kettle_on_thr189_anyOn_config.json
